# Evidence-literacy assistant: SFT + QLoRA
Runtime: Colab T4 GPU. Upload `handwritten_seed.jsonl`. Run top to bottom with `DRY_RUN=True` first (100 samples).

In [ ]:
!pip -q install -U transformers peft trl bitsandbytes datasets accelerate

In [ ]:
import torch, json, random
from datasets import load_dataset, Dataset
MODEL="Qwen/Qwen2.5-1.5B-Instruct"
DRY_RUN=True; N=100 if DRY_RUN else 1500; SEED=42
random.seed(SEED)
print(torch.cuda.get_device_name(0))

## Data: seed file (add your own 40-60 rows) + PubMedQA

In [ ]:
SYS="You are an evidence-literacy assistant. Classify the strength of evidence behind a claim and explain why. Educational only, not medical advice."
def fmt(r): return {"user":f"Claim: {r['claim']}","assistant":f"Evidence type: {r['evidence_type']}\nStrength: {r['strength']}\nWhy: {r['why']}\nWhat would strengthen it: {r['strengthen']}"}
rows=[fmt(json.loads(l)) for l in open("handwritten_seed.jsonl")]*5   # upsample the seed
try:
    pq=load_dataset("qiaojin/pubmed_qa","pqa_labeled",split="train")
    for r in pq.select(range(min(len(pq),N))):
        ctx=" ".join(r["context"]["contexts"])[:600]
        rows.append({"user":f"Question: {r['question']}\nAbstract excerpt: {ctx}\nDoes the evidence support yes, no or maybe?",
                     "assistant":f"Answer: {r['final_decision']}\nBasis: {r['long_answer'][:300]}\nNote: one abstract is a single source; check for systematic reviews."})
except Exception as e: print("PubMedQA failed, continuing with seed only:",e)
# TODO: add SciFact or health_fact here once you confirm the Hub ID loads
random.shuffle(rows); rows=rows[:N+50]; print(len(rows))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tok=AutoTokenizer.from_pretrained(MODEL); tok.pad_token=tok.pad_token or tok.eos_token
def to_text(r): return {"text":tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":r["user"]},{"role":"assistant","content":r["assistant"]}],tokenize=False)}
ds=Dataset.from_list(rows).map(to_text)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=bnb,device_map="auto")
print("GB:",torch.cuda.memory_allocated()/1e9)

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
model=prepare_model_for_kbit_training(model)
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=["q_proj","k_proj","v_proj","o_proj"],task_type="CAUSAL_LM")
kw=dict(output_dir="out",num_train_epochs=1,per_device_train_batch_size=4,gradient_accumulation_steps=4,learning_rate=2e-4,fp16=True,logging_steps=5,save_strategy="no",seed=SEED,dataset_text_field="text",report_to="none")
try: cfg=SFTConfig(max_length=512,**kw)
except TypeError: cfg=SFTConfig(max_seq_length=512,**kw)
try: trainer=SFTTrainer(model=model,args=cfg,train_dataset=ds,peft_config=lora,processing_class=tok)
except TypeError: trainer=SFTTrainer(model=model,args=cfg,train_dataset=ds,peft_config=lora,tokenizer=tok)
trainer.train()

In [ ]:
model.eval()
tests=["Claim: A new study found that a turmeric shot boosts immunity.","Claim: A systematic review of 30 trials found vitamin D does not prevent colds.","Claim: Five studies say coffee is good for you and four say it is bad, so it is good."]
for t in tests:
    ids=tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":t}],add_generation_prompt=True,return_tensors="pt",return_dict=True).to(model.device)
    out=model.generate(**ids,max_new_tokens=160,do_sample=False)
    print(t,"\n",tok.decode(out[0][ids["input_ids"].shape[1]:],skip_special_tokens=True),"\n"+"-"*40)

In [ ]:
trainer.model.save_pretrained("evidence-lora"); tok.save_pretrained("evidence-lora")  # screenshot loss curve and outputs